In [131]:
import pandas as pd
import numpy as np
import duckdb
import sklearn
import xgboost
import lightgbm
from unidecode import unidecode
import re
from rapidfuzz import process, fuzz


print("Entorno correcto")

Entorno correcto


In [113]:
%pip install Unidecode

Note: you may need to restart the kernel to use updated packages.


## **Data Source 1: Transfermarkt**

In [80]:
import duckdb

con = duckdb.connect("../data/raw/transfermarkt/transfermarkt-datasets.duckdb")

con.sql("SHOW TABLES")

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ appearances       │
│ club_games        │
│ clubs             │
│ competitions      │
│ countries         │
│ game_events       │
│ game_lineups      │
│ games             │
│ national_teams    │
│ player_valuations │
│ players           │
│ transfers         │
│ version           │
└───────────────────┘
       13 rows     

In [81]:
con.sql("DESCRIBE players")

┌──────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│             column_name              │ column_type │  null   │   key   │ default │  extra  │
│               varchar                │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ player_id                            │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ first_name                           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ last_name                            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ name                                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ last_season                          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ current_club_id                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_code                          │ VARCHAR  

In [82]:
con.sql("DESCRIBE appearances")

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ appearance_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ game_id                │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_id              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_club_id         │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_current_club_id │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date                   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ player_name            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ competition_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ yellow_cards           │ I

In [83]:
con.sql("DESCRIBE player_valuations")

┌─────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│             column_name             │ column_type │  null   │   key   │ default │  extra  │
│               varchar               │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ player_id                           │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ date                                │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ market_value_in_eur                 │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ current_club_name                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ current_club_id                     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_club_domestic_competition_id │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────────────────────────┴─────────────┴───────

In [84]:
con.sql("DESCRIBE transfers")

┌─────────────────────┬───────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │  column_type  │  null   │   key   │ default │  extra  │
│       varchar       │    varchar    │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼───────────────┼─────────┼─────────┼─────────┼─────────┤
│ player_id           │ INTEGER       │ YES     │ NULL    │ NULL    │ NULL    │
│ transfer_date       │ DATE          │ YES     │ NULL    │ NULL    │ NULL    │
│ transfer_season     │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ from_club_id        │ INTEGER       │ YES     │ NULL    │ NULL    │ NULL    │
│ to_club_id          │ INTEGER       │ YES     │ NULL    │ NULL    │ NULL    │
│ from_club_name      │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ to_club_name        │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ transfer_fee        │ DECIMAL(18,3) │ YES     │ NULL    │ NULL    │ NULL    │
│ market_value_in_eur │ DECIMAL(18,3) │ 

In [85]:
con.sql("SELECT * FROM players LIMIT 5")

┌───────────┬────────────┬──────────────┬────────────────────┬─────────────┬─────────────────┬────────────────────┬────────────────────┬───────────────┬────────────────────────┬─────────────────────┬────────────────┬────────────┬─────────┬──────────────┬──────────────────────────┬──────────────────────┬───────────────────────────────────────────────────────────────────────────────┬────────────────────┬─────────────────────┬──────────────────────────┬──────────────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────────────────────────────────────────┬─────────────────────┬─────────────────────────────┐
│ player_id │ first_name │  last_name   │        name        │ last_season │ current_club_id │    player_code     │  country_of_birth  │ city_of_birth │ country_of_citizenship │    date_of_birth    │  sub_position  │  position  │  foot   │ height_in_cm │ contract_expiration_date │      agent_name      │                     

In [86]:
con.sql("SELECT last_name FROM players WHERE last_season = '2025'")

┌────────────────────────┐
│       last_name        │
│        varchar         │
├────────────────────────┤
│ Guerrero               │
│ Milner                 │
│ Tsokanis               │
│ Hofmann                │
│ García                 │
│ Cristiano Ronaldo      │
│ Luiz Gustavo           │
│ Funes Mori             │
│ Carole                 │
│ Banega                 │
│   ·                    │
│   ·                    │
│   ·                    │
│ Banks                  │
│ Torres                 │
│ Marchiori              │
│ Lepasa                 │
│ Sepúlveda              │
│ Martel                 │
│ Moreno                 │
│ Ould-Chikh             │
│ de Lucas               │
│ Teles                  │
└────────────────────────┘
          ? rows        
  (>9999 rows, 20 shown) 

In [87]:
con.sql("""
SELECT 
    last_season,
    COUNT(*) AS players
FROM players
GROUP BY last_season
ORDER BY last_season DESC
""")

┌─────────────┬─────────┐
│ last_season │ players │
│   varchar   │  int64  │
├─────────────┼─────────┤
│ 2025        │   22292 │
│ 2024        │    6075 │
│ 2023        │    2092 │
│ 2022        │    1877 │
│ 2021        │    1817 │
│ 2020        │    1843 │
│ 2019        │    1566 │
│ 2018        │    2065 │
│ 2017        │    1661 │
│ 2016        │    1831 │
│ 2015        │    1651 │
│ 2014        │    1660 │
│ 2013        │    2061 │
│ 2012        │    1658 │
└─────────────┴─────────┘
  14 rows     2 columns

In [88]:
con.sql("""
SELECT 
    MIN(transfer_date) AS first_transfer,
    MAX(transfer_date) AS last_transfer,
    COUNT(*) AS total_transfers
FROM transfers
""")

┌────────────────┬───────────────┬─────────────────┐
│ first_transfer │ last_transfer │ total_transfers │
│      date      │     date      │      int64      │
├────────────────┼───────────────┼─────────────────┤
│ 1993-07-01     │ 2030-06-30    │          175165 │
└────────────────┴───────────────┴─────────────────┘

In [89]:
con.sql("""
SELECT 
    MIN(date) AS first_valuation,
    MAX(date) AS last_valuation,
    COUNT(*) AS total_valuations
FROM player_valuations
""")

┌─────────────────┬────────────────┬──────────────────┐
│ first_valuation │ last_valuation │ total_valuations │
│      date       │      date      │      int64       │
├─────────────────┼────────────────┼──────────────────┤
│ 2000-01-20      │ 2026-06-12     │           656301 │
└─────────────────┴────────────────┴──────────────────┘

In [90]:
con.sql("""
SELECT *
FROM transfers
WHERE transfer_date >= '2028-01-01'
ORDER BY transfer_date
LIMIT 20
""")

┌───────────┬───────────────┬─────────────────┬──────────────┬────────────┬─────────────────┬──────────────┬───────────────┬─────────────────────┬───────────────────┐
│ player_id │ transfer_date │ transfer_season │ from_club_id │ to_club_id │ from_club_name  │ to_club_name │ transfer_fee  │ market_value_in_eur │    player_name    │
│   int32   │     date      │     varchar     │    int32     │   int32    │     varchar     │   varchar    │ decimal(18,3) │    decimal(18,3)    │      varchar      │
├───────────┼───────────────┼─────────────────┼──────────────┼────────────┼─────────────────┼──────────────┼───────────────┼─────────────────────┼───────────────────┤
│    645842 │ 2028-02-02    │ 27/28           │         6505 │      19684 │ Gimcheon Sangmu │ Jeju SK      │         0.000 │          150000.000 │ Chan-gi An        │
│    677470 │ 2028-02-02    │ 27/28           │         6505 │       3535 │ Gimcheon Sangmu │ Ulsan HD     │         0.000 │          400000.000 │ Yool Heo          

##### Extracción de datos de la tabla players:

De la tabla `players`, se selecciona información de cada jugador que permita conocer mejor su contexto y características. Se opta por no incorporar directamente la variable `age`, ya que su valor depende del momento concreto en el que se mida. En su lugar, se conserva la fecha de nacimiento (`date_of_birth`), a partir de la cual podremos calcular posteriormente la edad correspondiente a cada jugador en función de la temporada analizada.

In [156]:
player_info = con.sql("""
    SELECT
        player_id,
        date_of_birth,
        country_of_citizenship,
        country_of_birth
    FROM players
""").df()

##### Extracción de datos de la tabla player_valuations:

Para incorporar la información de las diferentes valoraciones de mercado de cada jugador a lo largo de una temporada, se tiene en cuenta que estas pueden actualizarse en diferentes momentos del año. Por ello, se decide incorporar al dataset maestro 12 columnas correspondientes a cada mes del año. En cada registro, cada columna reflejará la valoración de mercado registrada para el jugador durante el mes correspondiente, manteniendo siempre la separación por temporadas. Las variables seguirán el orden cronológico de la temporada, desde `market_value_08` hasta `market_value_07`, correspondientes respectivamente a los meses de agosto a julio. De esta forma, el número asociado a cada variable permite identificar directamente el mes al que corresponde la valoración.

In [96]:
con.sql("""
SELECT *
FROM player_valuations
LIMIT 10
""")

┌───────────┬────────────┬─────────────────────┬─────────────────────┬─────────────────┬─────────────────────────────────────┐
│ player_id │    date    │ market_value_in_eur │  current_club_name  │ current_club_id │ player_club_domestic_competition_id │
│   int32   │    date    │        int32        │       varchar       │      int32      │               varchar               │
├───────────┼────────────┼─────────────────────┼─────────────────────┼─────────────────┼─────────────────────────────────────┤
│    405973 │ 2000-01-20 │              150000 │ Unknown             │            3057 │ BE1                                 │
│    342216 │ 2001-07-20 │              100000 │ Unknown             │            1241 │ SC1                                 │
│      3132 │ 2003-12-09 │              400000 │ Dynamo Kyiv         │             126 │ TR1                                 │
│      6893 │ 2003-12-15 │              900000 │ Galatasaray         │             984 │ GB1                   

In [101]:
con.sql("""
    CREATE OR REPLACE TABLE player_season_market_values AS

    WITH months AS (

        SELECT
            month_date,
            
            CASE
                WHEN EXTRACT(MONTH FROM month_date) >= 8
                    THEN CAST(EXTRACT(YEAR FROM month_date) AS VARCHAR)
                         || '-' ||
                         CAST(EXTRACT(YEAR FROM month_date) + 1 AS VARCHAR)
                ELSE
                    CAST(EXTRACT(YEAR FROM month_date) - 1 AS VARCHAR)
                         || '-' ||
                         CAST(EXTRACT(YEAR FROM month_date) AS VARCHAR)
            END AS season,

            EXTRACT(MONTH FROM month_date) AS month_number

        FROM generate_series(
            DATE '2023-08-01',
            DATE '2026-07-01',
            INTERVAL '1 month'
        ) AS t(month_date)
    ),

    player_months AS (

        SELECT
            p.player_id,
            m.month_date,
            m.season,
            m.month_number

        FROM (
            SELECT DISTINCT player_id
            FROM player_valuations
        ) p

        CROSS JOIN months m
    ),

    latest_values AS (

        SELECT
            pm.player_id,
            pm.season,
            pm.month_number,
            pv.market_value_in_eur,

            ROW_NUMBER() OVER (
                PARTITION BY
                    pm.player_id,
                    pm.season,
                    pm.month_number

                ORDER BY
                    pv.date DESC
            ) AS rn

        FROM player_months pm

        LEFT JOIN player_valuations pv
            ON pm.player_id = pv.player_id
            AND pv.date <= (
                pm.month_date
                + INTERVAL '1 month'
                - INTERVAL '1 day'
            )
    )

    SELECT
        p.name AS player,
        lv.player_id,
        lv.season,

        MAX(
            CASE
                WHEN lv.month_number = 8 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_08,

        MAX(
            CASE
                WHEN lv.month_number = 9 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_09,

        MAX(
            CASE
                WHEN lv.month_number = 10 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_10,

        MAX(
            CASE
                WHEN lv.month_number = 11 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_11,

        MAX(
            CASE
                WHEN lv.month_number = 12 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_12,

        MAX(
            CASE
                WHEN lv.month_number = 1 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_01,

        MAX(
            CASE
                WHEN lv.month_number = 2 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_02,

        MAX(
            CASE
                WHEN lv.month_number = 3 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_03,

        MAX(
            CASE
                WHEN lv.month_number = 4 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_04,

        MAX(
            CASE
                WHEN lv.month_number = 5 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_05,

        MAX(
            CASE
                WHEN lv.month_number = 6 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_06,

        MAX(
            CASE
                WHEN lv.month_number = 7 AND lv.rn = 1
                THEN lv.market_value_in_eur
            END
        ) AS market_value_07

    FROM latest_values lv

    LEFT JOIN players p
        ON lv.player_id = p.player_id

    GROUP BY
        p.name,
        lv.player_id,
        lv.season
""")

In [105]:
con.sql("""
    SELECT *
    FROM player_season_market_values
    WHERE player = 'Daniel Carvajal'
    ORDER BY season
""")

┌─────────────────┬───────────┬───────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┐
│     player      │ player_id │  season   │ market_value_08 │ market_value_09 │ market_value_10 │ market_value_11 │ market_value_12 │ market_value_01 │ market_value_02 │ market_value_03 │ market_value_04 │ market_value_05 │ market_value_06 │ market_value_07 │
│     varchar     │   int32   │  varchar  │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │
├─────────────────┼───────────┼───────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼───────────────

In [106]:
con.sql("""
    SELECT *
    FROM player_season_market_values
    WHERE market_value_08 IS NULL
    LIMIT 10
""")

┌───────────────────┬───────────┬───────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┐
│      player       │ player_id │  season   │ market_value_08 │ market_value_09 │ market_value_10 │ market_value_11 │ market_value_12 │ market_value_01 │ market_value_02 │ market_value_03 │ market_value_04 │ market_value_05 │ market_value_06 │ market_value_07 │
│      varchar      │   int32   │  varchar  │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │      int32      │
├───────────────────┼───────────┼───────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼───────

##### Extracción de datos de la tabla transfers:

Se procede a crear una nueva tabla con los datos que necesitamos de la tabla trasnfers, incluyendo la temporada, el nombre y el id del jugador. Se procede asimismo a la creación de nuevas variables a traves de estos datos, como el importe total de la suma de las transferencias de un jugador en una misma temporada, el importe máximo, el número de traspasos y una varaibla boobleana que indique si el jugador fue traspasado dicha temporada.

In [98]:
con.sql("""
SELECT *
FROM transfers
LIMIT 10
""")

┌───────────┬───────────────┬─────────────────┬──────────────┬────────────┬─────────────────┬──────────────┬───────────────┬─────────────────────┬───────────────────┐
│ player_id │ transfer_date │ transfer_season │ from_club_id │ to_club_id │ from_club_name  │ to_club_name │ transfer_fee  │ market_value_in_eur │    player_name    │
│   int32   │     date      │     varchar     │    int32     │   int32    │     varchar     │   varchar    │ decimal(18,3) │    decimal(18,3)    │      varchar      │
├───────────┼───────────────┼─────────────────┼──────────────┼────────────┼─────────────────┼──────────────┼───────────────┼─────────────────────┼───────────────────┤
│    467994 │ 2030-06-30    │ 25/26           │         5621 │        749 │ Reggiana        │ FC Empoli    │         0.000 │          700000.000 │ Luca Belardinelli │
│    645842 │ 2028-02-02    │ 27/28           │         6505 │      19684 │ Gimcheon Sangmu │ Jeju SK      │         0.000 │          150000.000 │ Chan-gi An        

In [99]:
con.sql("""
    CREATE OR REPLACE TABLE player_season_transfers AS

    SELECT
        p.name AS player,
        t.player_id,
        t.season,
        t.transferred,
        t.transfer_count,
        t.transfer_fee_total,
        t.transfer_fee_max

    FROM (
        SELECT
            player_id,

            CASE
                WHEN CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) >= 90
                    THEN '19' || SUBSTR(transfer_season, 1, 2)
                         || '-' ||
                         '19' || SUBSTR(transfer_season, 4, 2)
                ELSE
                    '20' || SUBSTR(transfer_season, 1, 2)
                    || '-' ||
                    '20' || SUBSTR(transfer_season, 4, 2)
            END AS season,

            TRUE AS transferred,
            COUNT(*) AS transfer_count,
            SUM(transfer_fee) AS transfer_fee_total,
            MAX(transfer_fee) AS transfer_fee_max

        FROM transfers

        GROUP BY
            player_id,
            transfer_season

    ) t

    LEFT JOIN players p
        ON t.player_id = p.player_id
""")

In [100]:
con.sql("""
SELECT *
FROM player_season_transfers
LIMIT 10
""")

┌───────────────────┬───────────┬───────────┬─────────────┬────────────────┬────────────────────┬──────────────────┐
│      player       │ player_id │  season   │ transferred │ transfer_count │ transfer_fee_total │ transfer_fee_max │
│      varchar      │   int32   │  varchar  │   boolean   │     int64      │   decimal(38,3)    │  decimal(18,3)   │
├───────────────────┼───────────┼───────────┼─────────────┼────────────────┼────────────────────┼──────────────────┤
│ Jordan Farr       │    565014 │ 2017-2018 │ true        │              2 │              0.000 │            0.000 │
│ Diego Rodríguez   │     90800 │ 2017-2018 │ true        │              2 │         625000.000 │       625000.000 │
│ Seung-kyum Im     │    292265 │ 2017-2018 │ true        │              2 │              0.000 │            0.000 │
│ Toni Martínez     │    302371 │ 2017-2018 │ true        │              2 │              0.000 │            0.000 │
│ Yuji Kajikawa     │    308651 │ 2017-2018 │ true        │     

##### Validación de nuevas tablas:

In [109]:
con.sql("""
    SELECT
        player_id,
        season,
        COUNT(*) AS n
    FROM player_season_transfers
    GROUP BY player_id, season
    HAVING COUNT(*) > 1
    ORDER BY n DESC
""")

┌───────────┬─────────┬───────┐
│ player_id │ season  │   n   │
│   int32   │ varchar │ int64 │
└───────────┴─────────┴───────┘
            0 rows           

In [110]:
con.sql("""
    SELECT
        player_id,
        season,
        COUNT(*) AS n
    FROM player_season_market_values
    GROUP BY player_id, season
    HAVING COUNT(*) > 1
    ORDER BY n DESC
""")

┌───────────┬─────────┬───────┐
│ player_id │ season  │   n   │
│   int32   │ varchar │ int64 │
└───────────┴─────────┴───────┘
            0 rows           

## **Data Source 2: Sofascore**

In [4]:
path = "../venv/Lib/site-packages/data/unified_player_stats.csv"

df = pd.read_csv(path)

df.shape

C:\Users\salit\AppData\Local\Temp\ipykernel_8244\768013171.py:3: DtypeWarning: Columns (0: pos) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


(22297, 149)

##### Mini-EDA unified_player_stats data

In [5]:
df.league.value_counts()

league
All Competitions            3361
UEFA Europa League          2740
UEFA Champions League       2477
England EFL Championship    2189
Spain La Liga               1783
Italy Serie A               1775
England Premier League      1668
Portugal Primeira Liga      1645
France Ligue 1              1618
Netherlands Eredivisie      1567
Germany Bundesliga          1474
Name: count, dtype: int64

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22297 entries, 0 to 22296
Columns: 149 entries, understat_id to npxg_overperformance
dtypes: bool(2), float64(131), int64(8), str(8)
memory usage: 26.9 MB


In [7]:
df["season"].value_counts().sort_index()

season
2023-2024    7209
2024-2025    7510
2025-2026    7578
Name: count, dtype: int64

In [8]:
df.head()

,understat_id,player,team,pos,games,minutes,goals,assists,shots,key_passes,...,xg_chain_per90,xg_buildup_per90,shots_per90,key_passes_per90,tackles_won_per90,interceptions_per90,big_chances_created_per90,dribbles_per90,xg_overperformance,npxg_overperformance
0,11295.0,David Datro Fofana,"1. FC Union Berlin,Burnley,Union Berlin",F S,31,1758,6,1,53,18,...,0.496,0.068,2.714,0.922,0.666,0.051,0.154,1.997,-2.99,-3.09
1,3699.0,Aissa Laidouni,"1. FC Union Berlin,Union Berlin",M S,29,1305,0,4,15,15,...,0.334,0.208,1.034,1.034,1.172,0.897,0.276,1.103,-0.98,-0.77
2,8017.0,Alex Kral,"1. FC Union Berlin,Union Berlin",D M S,30,1587,1,1,21,7,...,0.266,0.173,1.191,0.397,1.021,0.737,0.113,0.340,-0.78,-1.72
3,11236.0,Aljoscha Kemlein,"1. FC Union Berlin,Union Berlin",S,3,36,0,0,1,0,...,1.686,1.604,2.500,0.000,0.000,0.000,0.000,2.500,-0.03,-0.03
4,10751.0,Brenden Aaronson,"1. FC Union Berlin,Union Berlin",F M S,36,1337,2,2,22,22,...,0.500,0.226,1.480,1.480,1.211,0.336,0.269,2.355,-0.86,-0.38


In [9]:
df["league"].value_counts()

league
All Competitions            3361
UEFA Europa League          2740
UEFA Champions League       2477
England EFL Championship    2189
Spain La Liga               1783
Italy Serie A               1775
England Premier League      1668
Portugal Primeira Liga      1645
France Ligue 1              1618
Netherlands Eredivisie      1567
Germany Bundesliga          1474
Name: count, dtype: int64

In [10]:
df.groupby(["season", "league"]).size()

season     league                  
2023-2024  All Competitions            1070
           England EFL Championship     705
           England Premier League       569
           France Ligue 1               523
           Germany Bundesliga           494
           Italy Serie A                590
           Netherlands Eredivisie       516
           Portugal Primeira Liga       516
           Spain La Liga                596
           UEFA Champions League        721
           UEFA Europa League           909
2024-2025  All Competitions            1131
           England EFL Championship     736
           England Premier League       562
           France Ligue 1               542
           Germany Bundesliga           481
           Italy Serie A                599
           Netherlands Eredivisie       520
           Portugal Primeira Liga       562
           Spain La Liga                588
           UEFA Champions League        878
           UEFA Europa League           

In [11]:
df.groupby(["player", "season"]).size().describe()

count    15754.000000
mean         1.415323
std          0.821440
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          6.000000
dtype: float64

In [12]:
df["pos"].value_counts()

pos
D S        2419
M S        2144
F M S      1493
S          1121
D M S       898
F S         897
GK          631
D           455
D F M S     254
M           148
GK S        107
D M          58
F M          52
F            39
D F S        12
D F M         8
Name: count, dtype: int64

In [13]:
df["is_season_total"].value_counts(dropna=False)

is_season_total
False    18936
True      3361
Name: count, dtype: int64

In [14]:
df[df["is_season_total"] == True][
    ["player", "team", "league", "season", "games", "minutes"]
].head(20)

,player,team,league,season,games,minutes
0,David Datro Fofana,"1. FC Union Berlin,Burnley,Union Berlin",All Competitions,2023-2024,31,1758
1,Aissa Laidouni,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,29,1305
2,Alex Kral,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,30,1587
3,Aljoscha Kemlein,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,3,36
4,Brenden Aaronson,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,36,1337
5,Christopher Trimmel,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,29,1804
6,Danilho Doekhi,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,27,2420
7,Diogo Leite,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,38,3250
8,Frederik Rönnow,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,39,3510
9,Janik Haberer,"1. FC Union Berlin,Union Berlin",All Competitions,2023-2024,26,1588


In [15]:
df[df["is_season_total"] == False][
    ["player", "team", "league", "season", "games", "minutes"]
].head(20)

,player,team,league,season,games,minutes
3361,Alex Pritchard,Birmingham City,England EFL Championship,2023-2024,32,1455
3362,Andre Dozzell,Birmingham City,England EFL Championship,2023-2024,35,2613
3363,Brandon Khela,Birmingham City,England EFL Championship,2023-2024,1,9
3364,Cody Drameh,Birmingham City,England EFL Championship,2023-2024,29,2273
3365,Dion Sanderson,Birmingham City,England EFL Championship,2023-2024,37,3301
3366,Emanuel Aiwu,Birmingham City,England EFL Championship,2023-2024,24,1957
3367,Emmanuel Longelo,Birmingham City,England EFL Championship,2023-2024,17,1025
3368,Ethan Laird,Birmingham City,England EFL Championship,2023-2024,25,2001
3369,Gary Gardner,Birmingham City,England EFL Championship,2023-2024,16,248
3370,George Hall,Birmingham City,England EFL Championship,2023-2024,8,150


In [16]:
jugadores_total = set(
    df[df["is_season_total"] == True]["player"]
)

jugadores_competiciones = set(
    df[df["is_season_total"] == False]["player"]
)

jugadores_sin_total = jugadores_competiciones - jugadores_total

len(jugadores_sin_total)

6463

In [17]:
df_total = df[df["is_season_total"] == True].copy()

df_total.groupby("season")["player"].nunique()

season
2023-2024    1069
2024-2025    1131
2025-2026    1160
Name: player, dtype: int64

#### **Creación dataset del rendimiento en ligas domésticas - Big 5**

En este dataset se recoge el rendimiento de los jugadores pertenecientes a equipos de las cinco grandes ligas europeas (England Premier League, Spain La Liga, Germany Bundesliga, Italy Serie A y France Ligue 1) exclusivamente en sus respectivas competiciones domésticas. Por tanto, se excluye el rendimiento obtenido en competiciones europeas, como la UEFA Champions League o la UEFA Europa League. Este rendimiento se recoge de forma independiente en otro dataset, que contempla el rendimiento de los jugadores en el conjunto de las competiciones disputadas durante la temporada.

In [18]:
DOMESTIC_LEAGUES = [
    "England Premier League",
    "Spain La Liga",
    "Germany Bundesliga",
    "Italy Serie A",
    "France Ligue 1"
]

df_domestic = df[df["league"].isin(DOMESTIC_LEAGUES)].copy()

In [19]:
df_domestic.shape

(8318, 149)

#### Comprobación de valores de features df_domestic

Validación de consistencia de IDs

In [20]:
df_domestic[[
    "understat_id",
    "sofascore_id",
    "player",
    "team",
    "season",
    "league"
]].head(20)

,understat_id,sofascore_id,player,team,season,league
5550,5603.0,839410.0,Aaron Ramsdale,Arsenal,2023-2024,England Premier League
5551,7298.0,846036.0,Ben White,Arsenal,2023-2024,England Premier League
5552,7322.0,934235.0,Bukayo Saka,Arsenal,2023-2024,England Premier League
5553,847.0,44760.0,Cédric Soares,Arsenal,2023-2024,England Premier League
5554,9676.0,581310.0,David Raya,Arsenal,2023-2024,England Premier League
5555,5553.0,856714.0,Declan Rice,Arsenal,2023-2024,England Premier League
5556,6482.0,858194.0,Eddie Nketiah,Arsenal,2023-2024,England Premier League
5557,7230.0,867445.0,Emile Smith-Rowe,Arsenal,2023-2024,England Premier League
5558,11132.0,1401702.0,Ethan Nwaneri,Arsenal,2023-2024,England Premier League
5559,11007.0,904835.0,Fábio Vieira,Arsenal,2023-2024,England Premier League


In [21]:
multi_league = (
    df_domestic
    .groupby(["understat_id", "season"])["league"]
    .nunique()
    .sort_values(ascending=False)
)

multi_league[multi_league > 1].head(20)

understat_id  season   
8821.0        2023-2024    2
7095.0        2024-2025    2
5743.0        2023-2024    2
              2024-2025    2
2117.0        2024-2025    2
8981.0        2023-2024    2
638.0         2024-2025    2
8429.0        2023-2024    2
643.0         2023-2024    2
5329.0        2025-2026    2
8200.0        2024-2025    2
8393.0        2024-2025    2
239.0         2025-2026    2
5221.0        2025-2026    2
65.0          2023-2024    2
8700.0        2025-2026    2
9105.0        2023-2024    2
8812.0        2023-2024    2
7882.0        2023-2024    2
7236.0        2023-2024    2
Name: league, dtype: int64

In [22]:
multi_league[multi_league > 1].shape

(212,)

In [23]:
multi_league[multi_league > 1].head(5).index

MultiIndex([(8821.0, '2023-2024'),
            (7095.0, '2024-2025'),
            (5743.0, '2023-2024'),
            (5743.0, '2024-2025'),
            (2117.0, '2024-2025')],
           names=['understat_id', 'season'])

In [24]:
df_domestic.groupby(
    ["understat_id", "season"]
)["league"].nunique().gt(1).sum()

np.int64(212)

In [25]:
df_domestic.groupby(
    ["sofascore_id", "season"]
)["league"].nunique().gt(1).sum()

np.int64(215)

In [26]:
df_domestic[["understat_id", "sofascore_id"]].isna().sum()

understat_id    0
sofascore_id    0
dtype: int64

In [27]:
understat_multi = (
    df_domestic
    .groupby(["understat_id", "season"])["league"]
    .nunique()
)

sofascore_multi = (
    df_domestic
    .groupby(["sofascore_id", "season"])["league"]
    .nunique()
)

sofascore_multi[sofascore_multi > 1].index

MultiIndex([(      0.0, '2023-2024'),
            (      0.0, '2024-2025'),
            (      0.0, '2025-2026'),
            (  44614.0, '2024-2025'),
            (  45923.0, '2025-2026'),
            (  48480.0, '2025-2026'),
            (  78152.0, '2023-2024'),
            ( 132645.0, '2025-2026'),
            ( 146101.0, '2023-2024'),
            ( 158615.0, '2025-2026'),
            ...
            (1493689.0, '2024-2025'),
            (1500263.0, '2025-2026'),
            (1503757.0, '2025-2026'),
            (1514800.0, '2025-2026'),
            (1526628.0, '2025-2026'),
            (1536811.0, '2025-2026'),
            (1606792.0, '2024-2025'),
            (1823916.0, '2025-2026'),
            (1884145.0, '2025-2026'),
            (1926085.0, '2025-2026')],
           names=['sofascore_id', 'season'], length=215)

In [28]:
understat_multi_players = (
    df_domestic
    .groupby(["player", "season"])["league"]
    .nunique()
)

sofascore_multi_players = (
    df_domestic
    .groupby(["player", "season"])["league"]
    .nunique()
)

In [29]:
(understat_multi_players > 1).sum()

np.int64(226)

In [30]:
(sofascore_multi_players > 1).sum()

np.int64(226)

Sofascore merge failed

In [31]:
df_domestic._ss_merge_failed.value_counts()

_ss_merge_failed
False    8309
True        9
Name: count, dtype: int64

Sofascore ratings

In [32]:
df_domestic[
    [
        "player",
        "season",
        "league",
        "sofascore_rating",
        "sofascore_rating_total",
        "sofascore_rating_count"
    ]
].head(20)

,player,season,league,sofascore_rating,sofascore_rating_total,sofascore_rating_count
5550,Aaron Ramsdale,2023-2024,England Premier League,6.63,39.8,6.0
5551,Ben White,2023-2024,England Premier League,7.26,261.2,36.0
5552,Bukayo Saka,2023-2024,England Premier League,7.72,270.2,35.0
5553,Cédric Soares,2023-2024,England Premier League,6.63,19.9,3.0
5554,David Raya,2023-2024,England Premier League,6.85,219.2,32.0
5555,Declan Rice,2023-2024,England Premier League,7.43,282.4,38.0
5556,Eddie Nketiah,2023-2024,England Premier League,6.89,179.1,26.0
5557,Emile Smith-Rowe,2023-2024,England Premier League,6.67,246.7,37.0
5558,Ethan Nwaneri,2023-2024,England Premier League,6.90,6.9,1.0
5559,Fábio Vieira,2023-2024,England Premier League,6.92,69.2,10.0


Comprobación de nulos en df_domestic

In [33]:
missing = (
    df_domestic.isna()
    .mean()
    .sort_values(ascending=False)
)

for row in missing:
    if row > 0:
        print(row)

##### Clasificación de features

In [34]:
rate_cols = [
    col for col in df_domestic.columns
    if any(term in col.lower() for term in [
        "_per90",
        "_pct",
        "conversion",
        "frequency",
        "overperformance"
    ])
]
rate_cols.remove('ground_duels_won_pct')
rate_cols.remove('set_piece_conversion')
rate_cols.remove("scoring_frequency")
rate_cols

['tackles_won_pct',
 'aerials_won_pct',
 'dribbles_pct',
 'duels_won_pct',
 'pass_completion_pct',
 'long_balls_pct',
 'crosses_pct',
 'goal_conversion_pct',
 'pen_conversion_pct',
 'passes_opp_half_pct',
 'passes_own_half_pct',
 'chipped_passes_pct',
 'shots_inside_box_conversion_pct',
 'shots_outside_box_conversion_pct',
 'goals_per90',
 'assists_per90',
 'npg_per90',
 'xg_per90',
 'xag_per90',
 'npxg_per90',
 'xg_chain_per90',
 'xg_buildup_per90',
 'shots_per90',
 'key_passes_per90',
 'tackles_won_per90',
 'interceptions_per90',
 'big_chances_created_per90',
 'dribbles_per90',
 'xg_overperformance',
 'npxg_overperformance']

In [35]:
numeric_cols = df_domestic.select_dtypes(include="number").columns.tolist()

numeric_cols

['understat_id',
 'games',
 'minutes',
 'goals',
 'assists',
 'shots',
 'key_passes',
 'yellow_cards',
 'red_cards',
 'npg',
 'xg',
 'xag',
 'npxg',
 'xg_chain',
 'xg_buildup',
 'ninety_s',
 'sofascore_id',
 'sofascore_team_id',
 'sofascore_rating',
 'sofascore_rating_total',
 'sofascore_rating_count',
 'totw_appearances',
 'starts',
 'tackles',
 'tackles_won',
 'tackles_won_pct',
 'interceptions',
 'clearances',
 'blocked_shots',
 'outfield_blocks',
 'errors_leading_to_goal',
 'errors_leading_to_shot',
 'dribbled_past',
 'aerials_won',
 'aerials_won_pct',
 'aerials_lost',
 'dribbles_completed',
 'dribbles_pct',
 'dribbles_attempted',
 'ground_duels_won',
 'ground_duels_won_pct',
 'duels_won',
 'duels_won_pct',
 'duels_lost',
 'passes_total',
 'passes_completed',
 'passes_inaccurate',
 'pass_completion_pct',
 'passes_final_third',
 'passes_opp_half',
 'passes_own_half',
 'passes_opp_half_total',
 'passes_own_half_total',
 'long_balls_total',
 'long_balls_completed',
 'long_balls_pct',


In [36]:
non_numeric_cols = df_domestic.select_dtypes(exclude="number").columns.tolist()

non_numeric_cols

['player',
 'team',
 'pos',
 'league',
 'season',
 '_ss_merge_failed',
 'is_season_total',
 '_name_norm',
 'xg_source',
 'xag_source']

In [37]:
drop_cols = [
    "_ss_merge_failed",
    "_name_norm",
    "xg_source",
    "xag_source",
    "sofascore_team_id",
    "ground_duels_won_pct",
    "set_piece_conversion",
    "scoring_frequency"
]

In [38]:
categorical_cols = [
    "player",
    "team",
    "pos",
    "league",
    "season",
    "is_season_total"
]

In [39]:
id_cols = [
    "understat_id",
    "sofascore_id"
]

In [40]:
special_cols = [
    "sofascore_rating"
]

In [41]:
excluded_from_sum = (
    rate_cols
    + id_cols
    + special_cols
    + categorical_cols
    + drop_cols
)

In [42]:
sum_candidates = [
    col for col in numeric_cols
    if col not in excluded_from_sum
]

sum_candidates

['games',
 'minutes',
 'goals',
 'assists',
 'shots',
 'key_passes',
 'yellow_cards',
 'red_cards',
 'npg',
 'xg',
 'xag',
 'npxg',
 'xg_chain',
 'xg_buildup',
 'ninety_s',
 'sofascore_rating_total',
 'sofascore_rating_count',
 'totw_appearances',
 'starts',
 'tackles',
 'tackles_won',
 'interceptions',
 'clearances',
 'blocked_shots',
 'outfield_blocks',
 'errors_leading_to_goal',
 'errors_leading_to_shot',
 'dribbled_past',
 'aerials_won',
 'aerials_lost',
 'dribbles_completed',
 'dribbles_attempted',
 'ground_duels_won',
 'duels_won',
 'duels_lost',
 'passes_total',
 'passes_completed',
 'passes_inaccurate',
 'passes_final_third',
 'passes_opp_half',
 'passes_own_half',
 'passes_opp_half_total',
 'passes_own_half_total',
 'long_balls_total',
 'long_balls_completed',
 'crosses_total',
 'crosses_completed',
 'chipped_passes_total',
 'chipped_passes_completed',
 'pass_to_assist',
 'attempt_assists',
 'big_chances_created',
 'big_chances_missed',
 'goals_inside_box',
 'goals_outside_box

In [43]:
sum_cols = sum_candidates

In [44]:
classified_cols = (
    drop_cols
    + categorical_cols
    + id_cols
    + sum_cols
    + rate_cols
    + special_cols
)

unclassified_cols = [
    col for col in df_domestic.columns
    if col not in classified_cols
]

unclassified_cols

[]

In [45]:
categorical_aggregate_cols = [
    "team",
    "pos",
    "league"
]

In [46]:
group_cols = [
    "player",
    "season"
]

Para la creación del dataset procesado de rendimiento en las ligas domésticas de las Big 5, debido a la existencia de jugadores que han participado en varias ligas durante una misma temporada, se clasifican las variables en diferentes categorías para realizar una correcta agregación de los datos. Esta clasificación permite diferenciar entre variables acumulativas, ratios y porcentajes, y métricas de rendimiento normalizadas por cada 90 minutos.

##### Creación del pipeline para la agregación - player + season

In [47]:
# Función para agregar variables categóricas

def aggregate_unique(values):
    values = values.dropna().unique()

    if len(values) == 0:
        return np.nan

    if len(values) == 1:
        return values[0]

    return " | ".join(map(str, values))


In [48]:
# Función destinada a jugadores con 2 equipos de una misma liga en una misma temporada, para que se consideren como 2 equipos diferentes

def aggregate_unique_split(values):
    unique_values = set()

    for value in values.dropna():
        for item in str(value).split(","):
            item = item.strip()
            if item:
                unique_values.add(item)

    if len(unique_values) == 0:
        return np.nan

    return " | ".join(sorted(unique_values))

In [49]:
# Función para conteo de número de equipos por temporada y si cambió de equipo o no

def count_unique_split(values):
    unique_values = set()

    for value in values.dropna():
        for item in str(value).split(","):
            item = item.strip()
            if item:
                unique_values.add(item)

    return len(unique_values)

In [50]:
# Variables categóricas

categorical_aggregate_cols = [
    "team",
    "pos",
    "league"
]

In [51]:
# Relaciones para calcular porcentajes

percentage_map = {

    "tackles_won_pct": (
        "tackles_won",
        "tackles"
    ),

    "aerials_won_pct": (
        "aerials_won",
        ["aerials_won", "aerials_lost"]
    ),

    "dribbles_pct": (
        "dribbles_completed",
        "dribbles_attempted"
    ),

    "duels_won_pct": (
        "duels_won",
        ["duels_won", "duels_lost"]
    ),

    "pass_completion_pct": (
        "passes_completed",
        "passes_total"
    ),

    "long_balls_pct": (
        "long_balls_completed",
        "long_balls_total"
    ),

    "crosses_pct": (
        "crosses_completed",
        "crosses_total"
    ),

    "goal_conversion_pct": (
        "goals",
        "shots"
    ),

    "pen_conversion_pct": (
        "goals_penalty",
        "pens_taken"
    ),

    "passes_opp_half_pct": (
        "passes_opp_half",
        "passes_opp_half_total"
    ),

    "passes_own_half_pct": (
        "passes_own_half",
        "passes_own_half_total"
    ),

    "chipped_passes_pct": (
        "chipped_passes_completed",
        "chipped_passes_total"
    ),

    "shots_inside_box_conversion_pct": (
        "goals_inside_box",
        "shots_inside_box"
    ),

    "shots_outside_box_conversion_pct": (
        "goals_outside_box",
        "shots_outside_box"
    )
}

In [52]:
# Métricas por 90 minutos

per90_map = {

    "goals_per90": "goals",

    "assists_per90": "assists",

    "npg_per90": "npg",

    "xg_per90": "xg",

    "xag_per90": "xag",

    "npxg_per90": "npxg",

    "xg_chain_per90": "xg_chain",

    "xg_buildup_per90": "xg_buildup",

    "shots_per90": "shots",

    "key_passes_per90": "key_passes",

    "tackles_won_per90": "tackles_won",

    "interceptions_per90": "interceptions",

    "big_chances_created_per90": "big_chances_created",

    "dribbles_per90": "dribbles_completed"
}

In [53]:
# Función principal de agregación

def aggregate_player_season(df):
    
    df = df.drop(columns=drop_cols, errors="ignore").copy()

    # Agrupación por jugador y temporada

    grouped = df.groupby(
        ["player", "season"],
        sort=False
    )

    # 1. VARIABLES ACUMULATIVAS

    df_sum = (
        grouped[sum_cols]
        .sum(min_count=1)
    )

    # 2. VARIABLES CATEGÓRICAS

    df_team = (
        grouped["team"]
        .agg(aggregate_unique_split)
        .rename("team")
    )

    df_other_cat = (
        grouped[["pos", "league"]]
        .agg(aggregate_unique)
    )
    
    df_cat = pd.concat(
        [
            df_team,
            df_other_cat
        ],
        axis=1
    )

    # 3. IDENTIFICADORES

    df_ids = (
        grouped[id_cols]
        .first()
    )

    # 4. UNIÓN DE LOS BLOQUES

    result = pd.concat(
        [
            df_sum,
            df_cat,
            df_ids
        ],
        axis=1
    ).reset_index()

    # 5. NÚMERO DE EQUIPOS Y LIGAS

    counts = (
        grouped
        .agg(
            team_count=("team", count_unique_split),
            league_count=("league", "nunique")
        )
        .reset_index(drop=True)
    )
    
    result["team_count"] = counts["team_count"]
    result["league_count"] = counts["league_count"]

    # 6. INDICADORES DE CAMBIO

    result["changed_team"] = (
        result["team_count"] > 1
    ).astype(int)

    result["changed_league"] = (
        result["league_count"] > 1
    ).astype(int)

    # 7. SOFASCORE RATING

    result["sofascore_rating"] = np.where(
        result["sofascore_rating_count"] > 0,
        result["sofascore_rating_total"]
        / result["sofascore_rating_count"],
        np.nan
    )

    # 8. MÉTRICAS PORCENTUALES

    for col, (numerator, denominator) in percentage_map.items():

        if isinstance(denominator, list):
            denominator_value = (
                result[denominator]
                .sum(axis=1)
            )
        else:
            denominator_value = result[denominator]

        result[col] = np.where(
            denominator_value > 0,
            result[numerator]
            / denominator_value
            * 100,
            np.nan
        )

    # 9. MÉTRICAS POR 90 MINUTOS

    for rate_col, total_col in per90_map.items():

        result[rate_col] = np.where(
            result["minutes"] > 0,
            result[total_col]
            / result["minutes"]
            * 90,
            np.nan
        )

    # 10. XG OVERPERFORMANCE

    result["xg_overperformance"] = (
        result["goals"]
        - result["xg"]
    )

    result["npxg_overperformance"] = (
        result["npg"]
        - result["npxg"]
    )

    return result

##### Aplicación del pipeline de agregación para dataset de ligas domésticas - Big 5

In [54]:
df_domestic_processed = aggregate_player_season(df_domestic)

##### Comprobaciones tras aplicación del pipeline

In [55]:
# Cálculos de porcentajes coherentes

df_domestic_processed[["tackles_won", "tackles", "tackles_won_pct"]].head()

,tackles_won,tackles,tackles_won_pct
0,0.0,0.0,NaN
1,28.0,48.0,58.333333
2,35.0,66.0,53.030303
3,0.0,0.0,NaN
4,1.0,1.0,100.000000


In [56]:
# Una fila por jugador y temporada

df_domestic_processed.duplicated(
    subset=["player", "season"]
).sum()

np.int64(0)

In [57]:
# Comprobar que los jugadores que cambiaron de liga mantienen toda la información agregada

df_domestic_processed[
    df_domestic_processed["league_count"] > 1
][
    [
        "player",
        "season",
        "team",
        "league",
        "team_count",
        "league_count",
        "changed_team",
        "changed_league",
        "minutes",
        "goals"
    ]
].head(20)

,player,season,team,league,team_count,league_count,changed_team,changed_league,minutes,goals
10,Gabriel,2023-2024,Arsenal | Atletico Madrid | Valencia,England Premier League | Spain La Liga,3,2,1,1,4871,3
25,Bertrand Traoré,2023-2024,Aston Villa | Villarreal,England Premier League | Spain La Liga,2,2,1,1,575,1
39,Leander Dendoncker,2023-2024,Aston Villa | Napoli,England Premier League | Italy Serie A,2,2,1,1,133,1
64,Enes Ünal,2023-2024,Bournemouth | Getafe,England Premier League | Spain La Liga,2,2,1,1,416,2
65,Hamed Junior Traore,2023-2024,Bournemouth | Napoli,England Premier League | Italy Serie A,2,2,1,1,418,0
83,Romain Faivre,2023-2024,Bournemouth | Lorient,England Premier League | France Ligue 1,2,2,1,1,1473,5
116,Anssumane Fati,2023-2024,Barcelona | Brighton,England Premier League | Spain La Liga,2,2,1,1,482,2
120,Carlos Baleba,2023-2024,Brighton | Lille,England Premier League | France Ligue 1,2,2,1,1,1368,0
136,Mahmoud Dahoud,2023-2024,Brighton | VfB Stuttgart,England Premier League | Germany Bundesliga,2,2,1,1,622,1
152,David Datro Fofana,2023-2024,Burnley | Union Berlin,England Premier League | Germany Bundesliga,2,2,1,1,1560,5


In [58]:
df_domestic_processed[
    df_domestic_processed["team_count"] > 2
][
    [
        "player",
        "season",
        "team",
        "league",
        "team_count",
        "league_count",
        "changed_team",
        "changed_league"
    ]
].head(20)

,player,season,team,league,team_count,league_count,changed_team,changed_league
10,Gabriel,2023-2024,Arsenal | Atletico Madrid | Valencia,England Premier League | Spain La Liga,3,2,1,1
1356,Idrissa Gueye,2025-2026,Everton | Metz | Udinese,England Premier League | France Ligue 1 | Ital...,3,3,1,1


In [59]:
df_domestic_processed[
    df_domestic_processed["league_count"] > 2
][
    [
        "player",
        "season",
        "team",
        "league",
        "team_count",
        "league_count",
        "changed_team",
        "changed_league"
    ]
].head(20)

,player,season,team,league,team_count,league_count,changed_team,changed_league
1356,Idrissa Gueye,2025-2026,Everton | Metz | Udinese,England Premier League | France Ligue 1 | Ital...,3,3,1,1


In [60]:
df_domestic_processed[
    df_domestic_processed["player"] == "Idrissa Gueye"][
        ["season", "goals", "shots", "goal_conversion_pct"]
]

,season,goals,shots,goal_conversion_pct
242,2023-2024,4,26,15.384615
774,2024-2025,0,27,0.000000
1356,2025-2026,3,51,5.882353


##### Guardamos el dataset en .csv

In [61]:
df_domestic_processed.to_csv(
    "../data/processed_data/domestic_player_seasons.csv",
    index=False
)

#### **Creación dataset del rendimiento en todas las competiciones - All Competitions**

In [62]:
df["league"].value_counts()

league
All Competitions            3361
UEFA Europa League          2740
UEFA Champions League       2477
England EFL Championship    2189
Spain La Liga               1783
Italy Serie A               1775
England Premier League      1668
Portugal Primeira Liga      1645
France Ligue 1              1618
Netherlands Eredivisie      1567
Germany Bundesliga          1474
Name: count, dtype: int64

In [63]:
ALL_COMPS = [comp for comp in df["league"].value_counts().index if comp != 'All Competitions']

In [64]:
df_all_comps = df_domestic = df[df["league"].isin(ALL_COMPS)].copy()

##### Aplicación del pipeline de agregación a df_all_comps

In [65]:
df_all_comps_processed = aggregate_player_season(df_all_comps)

##### Comprobaciones tras aplicación del pipeline

In [ ]:
df_all_comps_processed[df_all_comps_processed['player'] == 'Andre Dozzell']

,player,season,games,minutes,goals,assists,shots,key_passes,yellow_cards,red_cards,...,xg_chain_per90,xg_buildup_per90,shots_per90,key_passes_per90,tackles_won_per90,interceptions_per90,big_chances_created_per90,dribbles_per90,xg_overperformance,npxg_overperformance
1,Andre Dozzell,2023-2024,35,2613,3,1,23,30,4,1,...,0.0,0.0,0.792193,1.033295,0.861079,0.826636,0.137773,0.275545,1.86,0.0
1146,Andre Dozzell,2024-2025,39,2826,2,1,16,17,11,0,...,0.0,0.0,0.509554,0.541401,1.019108,0.796178,0.127389,0.159236,-0.47,0.0
1843,Andre Dozzell,2025-2026,37,3157,2,1,16,25,8,0,...,0.0,0.0,0.456129,0.712702,0.598670,0.712702,0.085524,0.456129,1.05,0.0


In [72]:
df_all_comps_processed[(df_all_comps_processed['league_count'] < 2) & (df_all_comps_processed["team"] == 'Real Madrid')].head(10)

,player,season,games,minutes,goals,assists,shots,key_passes,yellow_cards,red_cards,...,xg_chain_per90,xg_buildup_per90,shots_per90,key_passes_per90,tackles_won_per90,interceptions_per90,big_chances_created_per90,dribbles_per90,xg_overperformance,npxg_overperformance
11999,Alvaro Rodríguez,2023-2024,1,1,0,0,0,0,0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,180.000000,0.000000,0.000000
12002,Arda Güler,2023-2024,10,365,6,0,15,2,1,0,...,1.226164,0.812662,3.698630,0.493151,0.986301,0.493151,0.000000,2.712329,4.008871,4.008871
12003,Aurelien Tchouameni,2023-2024,27,1989,3,1,27,10,6,0,...,0.648968,0.605354,1.221719,0.452489,1.040724,1.266968,0.045249,0.271493,1.871517,1.871517
12004,Brahim Diaz,2023-2024,31,1548,8,6,38,30,0,0,...,1.030254,0.413809,2.209302,1.744186,1.104651,0.290698,0.523256,3.197674,0.935330,0.935330
12006,Daniel Carvajal,2023-2024,28,2191,4,3,18,21,3,1,...,0.730339,0.558492,0.739388,0.862620,0.451848,0.246463,0.205386,0.082154,1.415679,1.415679
12008,Eder Militão,2023-2024,10,498,0,0,2,0,0,0,...,0.587334,0.584711,0.361446,0.000000,0.542169,0.722892,0.000000,0.000000,-0.080279,-0.080279
12013,Gonzalo García,2023-2024,2,10,0,0,1,0,0,0,...,0.488311,0.000000,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.054257,-0.054257
12016,Kepa,2023-2024,14,1194,0,0,0,0,1,0,...,0.278204,0.278204,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
12018,Luka Modric,2023-2024,32,1673,2,6,25,55,2,0,...,0.918280,0.777390,1.344889,2.958757,0.806934,1.129707,0.645547,1.129707,0.765528,0.765528
12019,Mario Martín,2023-2024,2,9,0,0,0,0,0,0,...,0.000000,0.000000,0.000000,0.000000,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [77]:
df[df['player'] == 'Dani Carvajal'][['player', 'season', 'league', 'team']]

,player,season,league,team
17625,Dani Carvajal,2023-2024,UEFA Champions League,Real Madrid
18498,Dani Carvajal,2024-2025,UEFA Champions League,Real Madrid
19385,Dani Carvajal,2025-2026,UEFA Champions League,Real Madrid


In [ ]:
df_all_comps_processed[df_all_comps_processed['player'] == 'Arda Güler'][['player', 'season', 'league', 'team']]

Durante la revisión del dataset de competiciones se detectaron algunos casos en los que determinados jugadores no presentan información de todas las competiciones disputadas durante una temporada. Tras comprobar estos casos directamente sobre el dataset original, se confirmó que la ausencia de registros procede de la fuente de datos y no del proceso de filtrado o agregación aplicado durante el procesamiento.

Por ejemplo, Dani Carvajal únicamente dispone de registros correspondientes a la UEFA Champions League en las tres temporadas analizadas, sin que existan en la fuente registros correspondientes a La Liga. En el caso de Arda Güler, sí aparecen registros de La Liga durante las tres temporadas y de Champions League durante 2024-2025 y 2025-2026, pero no existe un registro de Champions League para 2023-2024.

Por este motivo, el pipeline mantiene únicamente la información disponible en la fuente original y no se introducen manualmente datos de competiciones ausentes. De esta forma, se preserva la trazabilidad y consistencia del dataset, evitando incorporar información externa de forma arbitraria. Estos casos deberán tenerse en cuenta posteriormente como una limitación de cobertura de la fuente y en el tratamiento de valores ausentes durante las fases de construcción del dataset maestro y modelado.


##### Guardamos el dataset en formato .csv

In [78]:
df_all_comps_processed.to_csv(
    "../data/processed_data/all_competitions_player_seasons.csv",
    index=False
)

## **Creación del dataset maestro**

##### Normalización y matching de nombres entre las diferentes tablas

In [111]:
players_sofascore = (
    df_all_comps_processed[["player"]]
    .drop_duplicates()
    .copy()
)

In [112]:
players_transfermarkt = con.sql("""
    SELECT
        player_id,
        name
    FROM players
""").df()

In [115]:
def normalize_name(name):
    
    if pd.isna(name):
        return np.nan
    
    name = unidecode(str(name)).lower().strip()
    
    name = re.sub(r"[^a-z0-9\s]", "", name)
    name = re.sub(r"\s+", " ", name)
    
    return name

In [116]:
players_sofascore["name_norm"] = (
    players_sofascore["player"]
    .apply(normalize_name)
)

players_transfermarkt["name_norm"] = (
    players_transfermarkt["name"]
    .apply(normalize_name)
)

In [117]:
player_mapping = players_sofascore.merge(
    players_transfermarkt,
    on="name_norm",
    how="left",
    suffixes=("_sofascore", "_transfermarkt")
)

In [118]:
player_mapping["player_id"].notna().value_counts()

player_id
True     8136
False    1050
Name: count, dtype: int64

In [119]:
player_mapping["player_id"].notna().mean() * 100

np.float64(88.56956237753103)

In [123]:
unmatched_players = (
    player_mapping[
        player_mapping["player_id"].isna()
    ]
    [["player", "name_norm"]]
    .sort_values("player")
)

unmatched_players.head(515)

,player,name_norm
449,Aaron Drewe,aaron drewe
7016,Aarón,aaron
6741,Abdelkabir Abqar,abdelkabir abqar
6825,Abdellah Raihani,abdellah raihani
6742,Abderrahmane Rebbach,abderrahmane rebbach
...,...,...
9132,José Morante,jose morante
4382,José Reina,jose reina
6671,José Tavares,jose tavares
1211,Jovon Makama,jovon makama


In [121]:
len(unmatched_players)

1050

In [122]:
players_transfermarkt[
    players_transfermarkt["name_norm"].duplicated(keep=False)
].sort_values("name_norm")

,player_id,name,name_norm
10010,136581,Aaron Martin,aaron martin
16138,251878,Aarón Martín,aaron martin
4355,50057,Aaron Ramsey,aaron ramsey
33270,646658,Aaron Ramsey,aaron ramsey
3654,42941,Abdou Traoré,abdou traore
...,...,...,...
39527,879236,Zé Vítor,ze vitor
9752,131161,Zeca,zeca
19936,325196,Zeca,zeca
6342,72860,Zezinho,zezinho


In [125]:
df_all_comps_processed.columns.tolist()

['player',
 'season',
 'games',
 'minutes',
 'goals',
 'assists',
 'shots',
 'key_passes',
 'yellow_cards',
 'red_cards',
 'npg',
 'xg',
 'xag',
 'npxg',
 'xg_chain',
 'xg_buildup',
 'ninety_s',
 'sofascore_rating_total',
 'sofascore_rating_count',
 'totw_appearances',
 'starts',
 'tackles',
 'tackles_won',
 'interceptions',
 'clearances',
 'blocked_shots',
 'outfield_blocks',
 'errors_leading_to_goal',
 'errors_leading_to_shot',
 'dribbled_past',
 'aerials_won',
 'aerials_lost',
 'dribbles_completed',
 'dribbles_attempted',
 'ground_duels_won',
 'duels_won',
 'duels_lost',
 'passes_total',
 'passes_completed',
 'passes_inaccurate',
 'passes_final_third',
 'passes_opp_half',
 'passes_own_half',
 'passes_opp_half_total',
 'passes_own_half_total',
 'long_balls_total',
 'long_balls_completed',
 'crosses_total',
 'crosses_completed',
 'chipped_passes_total',
 'chipped_passes_completed',
 'pass_to_assist',
 'attempt_assists',
 'big_chances_created',
 'big_chances_missed',
 'goals_inside_box

In [127]:
players_columns = con.sql("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'players'
""").df()

players_columns

,column_name
0,player_id
1,first_name
2,last_name
3,name
4,last_season
5,current_club_id
6,player_code
7,country_of_birth
8,city_of_birth
9,country_of_citizenship


In [129]:
con.sql("""
    SELECT
        player_id,
        name,
        date_of_birth
    FROM players
    WHERE lower(name) IN ('aaron martin', 'aaron ramsey')
    ORDER BY name, date_of_birth
""")

┌───────────┬──────────────┬─────────────────────┐
│ player_id │     name     │    date_of_birth    │
│   int32   │   varchar    │      timestamp      │
├───────────┼──────────────┼─────────────────────┤
│    136581 │ Aaron Martin │ 1989-09-29 00:00:00 │
│     50057 │ Aaron Ramsey │ 1990-12-26 00:00:00 │
│    646658 │ Aaron Ramsey │ 2003-01-21 00:00:00 │
└───────────┴──────────────┴─────────────────────┘

In [130]:
%pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [132]:
players_sofascore = (
    df_all_comps_processed[
        ["sofascore_id", "player"]
    ]
    .drop_duplicates()
    .copy()
)

players_sofascore["name_norm"] = (
    players_sofascore["player"]
    .apply(normalize_name)
)

In [133]:
players_transfermarkt = con.sql("""
    SELECT
        player_id,
        name,
        date_of_birth
    FROM players
""").df()

players_transfermarkt["name_norm"] = (
    players_transfermarkt["name"]
    .apply(normalize_name)
)

In [134]:
tm_name_counts = (
    players_transfermarkt
    .groupby("name_norm")["player_id"]
    .nunique()
    .rename("tm_player_count")
)

players_transfermarkt = players_transfermarkt.merge(
    tm_name_counts,
    on="name_norm",
    how="left"
)

In [135]:
player_mapping = players_sofascore.merge(
    players_transfermarkt[
        [
            "player_id",
            "name",
            "name_norm",
            "date_of_birth",
            "tm_player_count"
        ]
    ],
    on="name_norm",
    how="left",
    suffixes=("_sofascore", "_transfermarkt")
)

In [139]:
player_mapping["match_method"] = (
    player_mapping["name_norm"]
    .where(
        player_mapping["tm_player_count"] == 1
    )
)

In [140]:
player_mapping["match_method"].value_counts(dropna=False)

match_method
NaN                2081
famara diedhiou       2
junior firpo          2
george evans          2
milutin osmajic       2
                   ... 
luca weinhandl        1
matteo bignetti       1
maurice malone        1
joao aragao           1
yanik spalt           1
Name: count, Length: 7000, dtype: int64

In [141]:
matched_players = player_mapping[
    player_mapping["match_method"].notna()
].copy()

In [142]:
unmatched_players = player_mapping[
    player_mapping["match_method"].isna()
].copy()

In [143]:
len(players_sofascore), len(matched_players), len(unmatched_players)

(8635, 7217, 2081)

In [159]:
unmatched_players[
    ["sofascore_id", "player", "name_norm"]
].head(10)

,sofascore_id,player,name_norm
9,1136413.0,George Hall,george hall
21,354802.0,Marc Roberts,marc roberts
22,1020642.0,Marcel Oakley,marcel oakley
26,102280.0,Scott Hogan,scott hogan
27,881307.0,Seung Ho Paik,seung ho paik
33,903166.0,Aynsley Pears,aynsley pears
35,828620.0,Callum Brittain,callum brittain
36,1133480.0,Connor O'Riordan,connor oriordan
38,802719.0,Dominic Hyam,dominic hyam
39,1141108.0,Harry Leonard,harry leonard


In [145]:
tm_names = (
    players_transfermarkt["name_norm"]
    .dropna()
    .unique()
    .tolist()
)

In [146]:
def find_name_candidates(name, choices, limit=3):
    
    return process.extract(
        name,
        choices,
        scorer=fuzz.ratio,
        limit=limit
    )

In [147]:
unmatched_players["name_candidates"] = (
    unmatched_players["name_norm"]
    .apply(lambda x: find_name_candidates(x, tm_names))
)

In [148]:
unmatched_players[
    ["player", "name_candidates"]
].head(20)

,player,name_candidates
9,George Hall,"[(george saville, 80.0, 10109), (george mells,..."
21,Marc Roberts,"[(marc rebes, 81.81818181818181, 14600), (patr..."
22,Marcel Oakley,"[(marcelo morales, 78.57142857142857, 37647), ..."
26,Scott Hogan,"[(scott dann, 76.19047619047619, 5651), (scott..."
27,Seung Ho Paik,"[(seungho paik, 96.0, 17486), (seungho park, 8..."
33,Aynsley Pears,"[(ashley barnes, 69.23076923076923, 5685), (al..."
35,Callum Brittain,"[(callum tapping, 75.86206896551724, 11930), (..."
36,Connor O'Riordan,"[(connor ronan, 81.4814814814815, 20206), (con..."
38,Dominic Hyam,"[(dominic tan, 78.26086956521739, 26459), (dom..."
39,Harry Leonard,"[(marc leonard, 80.0, 25905), (mark leonard, 8..."


In [149]:
len(set(matched_players) & set(unmatched_players))

8

In [150]:
player_mapping["tm_player_count"].value_counts().sort_index()

tm_player_count
1.0     7217
2.0      472
3.0      201
4.0      136
5.0       60
6.0       24
7.0       14
8.0       72
11.0      11
13.0      13
14.0      28
Name: count, dtype: int64

In [154]:
player_mapping["tm_player_count"].value_counts(True)

tm_player_count
1.0     0.875000
2.0     0.057226
3.0     0.024370
4.0     0.016489
8.0     0.008729
5.0     0.007274
14.0    0.003395
6.0     0.002910
7.0     0.001697
13.0    0.001576
11.0    0.001334
Name: proportion, dtype: float64

In [151]:
player_mapping[
    player_mapping["tm_player_count"] > 1
][
    ["player", "name_norm", "tm_player_count"]
].sort_values("name_norm").head(20)

,player,name_norm,tm_player_count
4015,Aarón Martín,aaron martin,2.0
4016,Aarón Martín,aaron martin,2.0
84,Aaron Ramsey,aaron ramsey,2.0
85,Aaron Ramsey,aaron ramsey,2.0
1177,Aaron Ramsey,aaron ramsey,2.0
1178,Aaron Ramsey,aaron ramsey,2.0
4705,Abdoulaye Camara,abdoulaye camara,2.0
4706,Abdoulaye Camara,abdoulaye camara,2.0
2939,Abdoulaye Faye,abdoulaye faye,2.0
2940,Abdoulaye Faye,abdoulaye faye,2.0


Dada la falta de disponibilidad de un identificador común entre ambas fuentes de datos (SofaScore y Transfermarkt), se opta por utilizar el nombre de los jugadores como variable de enlace entre ambas fuentes.

El enlace mediante el nombre presenta ciertos problemas de coincidencia derivados de diferencias en la acentuación, diminutivos, nombres compuestos o distintas formas de escritura. Por este motivo, en primer lugar se normalizan los nombres de los jugadores presentes en ambas fuentes y, posteriormente, se buscan coincidencias sobre los nombres normalizados para establecer la correspondencia entre registros y poder integrar ambas fuentes de datos.

Se obtienen coincidencias únicas y exactas para el 87,5 % de los jugadores, permitiendo establecer un match directo entre ambas fuentes. Para el 12,5 % restante, se observa que existen nombres que corresponden a varios jugadores diferentes dentro de Transfermarkt.

Ante la ausencia de la fecha de nacimiento en la fuente de SofaScore, no resulta posible utilizar esta variable como criterio adicional para determinar de forma fiable si los registros con un mismo nombre corresponden al mismo jugador o a jugadores diferentes. Por este motivo, se decide no establecer coincidencias en aquellos casos ambiguos, evitando asignar información de Transfermarkt a un jugador incorrecto.

De esta forma, se prioriza la fiabilidad y trazabilidad de los datos frente a maximizar el número de coincidencias. Los jugadores sin una correspondencia inequívoca se mantienen en el dataset de SofaScore, pero no se incorporan datos procedentes de Transfermarkt para ellos.

Se plantea también la posibilidad de utilizar el equipo como criterio adicional para resolver algunas de estas coincidencias. Sin embargo, esta alternativa presenta problemas similares, debido a que los nombres de los equipos pueden aparecer escritos de diferentes formas entre ambas fuentes y, además, los jugadores pueden pertenecer a diferentes equipos a lo largo de una misma temporada. Por este motivo, se decide no utilizar el equipo como criterio de matching automático.

In [162]:
player_mapping["match_method"].value_counts(dropna=False)

match_method
NaN                2081
famara diedhiou       2
junior firpo          2
george evans          2
milutin osmajic       2
                   ... 
luca weinhandl        1
matteo bignetti       1
maurice malone        1
joao aragao           1
yanik spalt           1
Name: count, Length: 7000, dtype: int64

In [163]:
# Nos quedamos solo con los jugadores que han tenido match a través del nombre nomrmalzado (norm_name)

matched_mapping = (
    player_mapping[
        player_mapping["tm_player_count"] == 1
    ]
    [["sofascore_id", "player", "player_id"]]
    .drop_duplicates()
)

##### Añadimos los datos de player_info

In [164]:
player_info_mapping = matched_mapping.merge(
    player_info,
    on="player_id",
    how="left"
)

In [165]:
player_info_mapping["birth_year"] = (
    pd.to_datetime(
        player_info_mapping["date_of_birth"],
        errors="coerce"
    )
    .dt.year
)

In [166]:
player_info_mapping

,sofascore_id,player,player_id,date_of_birth,country_of_citizenship,country_of_birth,birth_year
0,180009.0,Alex Pritchard,200777.0,1993-05-03,England,England,1993.0
1,826133.0,Andre Dozzell,346467.0,1999-05-02,England,England,1999.0
2,1161806.0,Brandon Khela,921735.0,2005-01-19,England,NaN,2005.0
3,991577.0,Cody Drameh,531957.0,2001-12-08,England,England,2001.0
4,970407.0,Dion Sanderson,495993.0,1999-12-15,England,England,1999.0
...,...,...,...,...,...,...,...
7212,877845.0,Maurice Malone,405568.0,2000-08-17,Germany,Germany,2000.0
7213,1961097.0,João Aragão,1075884.0,2008-03-24,Portugal,Portugal,2008.0
7214,1464641.0,Chema Andrés,948279.0,2005-04-25,Spain,Spain,2005.0
7215,1427148.0,Lazar Jovanović,897505.0,2006-11-30,Serbia,Serbia,2006.0


Limpieza de mapping player_info

In [167]:
len(matched_mapping)

7217

In [176]:
player_info_mapping[player_info_mapping['player_id'].duplicated()]

,sofascore_id,player,player_id,date_of_birth,country_of_citizenship,country_of_birth,birth_year
627,2032373.0,George Evans,181498.0,1994-12-13,England,England,1994.0
818,1130909.0,Cameron Humphreys,277723.0,1998-08-22,England,England,1998.0
1050,1149855.0,Mark O’Mahony,943549.0,2005-01-14,Ireland,Ireland,2005.0
1092,285989.0,Jairo Riedewald,241481.0,1996-09-09,Netherlands,Netherlands,1996.0
1164,1006189.0,Issa Kabore,649452.0,2001-05-12,Burkina Faso,Burkina Faso,2001.0
...,...,...,...,...,...,...,...
7169,1152139.0,Angel Ortiz,933908.0,2004-07-25,Spain,Spain,2004.0
7172,964983.0,Álvaro Valles,417913.0,1997-07-25,Spain,Spain,1997.0
7206,796032.0,Emir Karić,286596.0,1997-06-09,Bosnia-Herzegovina,Austria,1997.0
7214,1464641.0,Chema Andrés,948279.0,2005-04-25,Spain,Spain,2005.0


In [172]:
player_info_mapping[
    player_info_mapping["player_id"].duplicated(keep=False)
].sort_values("player_id")

,sofascore_id,player,player_id,date_of_birth,country_of_citizenship,country_of_birth,birth_year
6090,15466.0,Luka Modrić,27992.0,1985-09-09,Croatia,Jugoslawien (SFR),1985.0
3642,15466.0,Luka Modric,27992.0,1985-09-09,Croatia,Jugoslawien (SFR),1985.0
6845,14990.0,Edin Džeko,28396.0,1986-03-17,Bosnia-Herzegovina,Jugoslawien (SFR),1986.0
3676,14990.0,Edin Dzeko,28396.0,1986-03-17,Bosnia-Herzegovina,Jugoslawien (SFR),1986.0
6734,15479.0,Łukasz Fabiański,29692.0,1985-04-18,Poland,Poland,1985.0
...,...,...,...,...,...,...,...
2311,1500263.0,Aaron Anselmino,1145504.0,2005-04-29,Argentina,Argentina,2005.0
5831,1953923.0,Selton Sánchez,1312698.0,2007-02-20,Spain,Spain,2007.0
6332,1953923.0,Selton Sanchez,1312698.0,2007-02-20,Spain,Spain,2007.0
6480,1966890.0,Hugo López,1331450.0,2007-03-04,Spain,Spain,2007.0


In [221]:
player_info_mapping.duplicated().sum()

np.int64(0)

In [180]:
player_info_mapping = (
    player_info_mapping
    .drop_duplicates(subset="player_id", keep="first")
    .reset_index(drop=True)
)

In [182]:
player_info_mapping["player_id"].duplicated().sum()

np.int64(0)

In [220]:
player_info_mapping[
    player_info_mapping["player_id"].duplicated(keep=False)
]["player_id"].nunique()

0

In [222]:
player_info_mapping[
    player_info_mapping["sofascore_id"].duplicated(keep=False)
].sort_values("sofascore_id")

,sofascore_id,player,player_id,date_of_birth,country_of_citizenship,country_of_birth,birth_year
5633,175753.0,Mariano,54155.0,1986-06-23,Brazil,Brazil,1986.0
6033,175753.0,Mariano Díaz,225020.0,1993-08-01,Dominican Republic,Spain,1993.0
5672,831253.0,Antonio Martínez,1186240.0,2006-04-09,Spain,Spain,2006.0
4707,831253.0,Toni Martínez,302371.0,1997-06-30,Spain,Spain,1997.0
5882,942836.0,Haissem Hassan,620164.0,2002-02-08,Egypt,France,2002.0
5894,942836.0,Rahim Alhassane,929899.0,2002-01-01,Niger,Nigeria,2002.0
6146,1046795.0,Savinho,743591.0,2004-04-10,Brazil,Brazil,2004.0
1385,1046795.0,Sávio,607093.0,1995-05-26,Brazil,Brazil,1995.0
2246,1107439.0,Ali Youssif,944724.0,2001-07-09,Libya,Libya,2001.0
2245,1107439.0,Ali Youssef,501131.0,2000-08-05,Tunisia,Sweden,2000.0


In [223]:
ambiguous_player_ids = (
    player_info_mapping[
        player_info_mapping["sofascore_id"].duplicated(keep=False)
    ]["sofascore_id"]
    .unique()
)

player_info_mapping = player_info_mapping[
    ~player_info_mapping["sofascore_id"].isin(ambiguous_player_ids)
].copy()

In [224]:
player_info_mapping["sofascore_id"].duplicated().sum()

np.int64(0)

Limpieza de mapping market_values

In [185]:
market_values = con.sql("""
    SELECT *
    FROM player_season_market_values
""").df()

In [186]:
market_values_mapping = market_values.merge(
    matched_mapping[
        [
            "sofascore_id",
            "player_id"
        ]
    ],
    on="player_id",
    how="inner"
)

In [187]:
market_values_mapping = market_values_mapping[
    [
        "sofascore_id",
        "season",
        "market_value_08",
        "market_value_09",
        "market_value_10",
        "market_value_11",
        "market_value_12",
        "market_value_01",
        "market_value_02",
        "market_value_03",
        "market_value_04",
        "market_value_05",
        "market_value_06",
        "market_value_07"
    ]
].copy()

In [188]:
market_values_mapping[
    ["sofascore_id", "season"]
].duplicated().sum()

np.int64(636)

In [189]:
market_values_mapping[
    market_values_mapping.duplicated(
        subset=["sofascore_id", "season"],
        keep=False
    )
].sort_values(["sofascore_id", "season"])

,sofascore_id,season,market_value_08,market_value_09,market_value_10,market_value_11,market_value_12,market_value_01,market_value_02,market_value_03,market_value_04,market_value_05,market_value_06,market_value_07
17917,7635.0,2023-2024,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,3500000,3500000
17918,7635.0,2023-2024,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,3500000,3500000
973,7635.0,2024-2025,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000
974,7635.0,2024-2025,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000
20351,7635.0,2025-2026,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5683,2059250.0,2023-2024,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4844,2059250.0,2024-2025,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4845,2059250.0,2024-2025,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6360,2059250.0,2025-2026,<NA>,<NA>,<NA>,<NA>,100000,100000,100000,100000,100000,700000,700000,700000


In [190]:
market_values_mapping[
    market_values_mapping.duplicated(
        subset=["sofascore_id", "season"],
        keep=False
    )
].shape

(1272, 14)

In [192]:
market_values_mapping[
    market_values_mapping.duplicated(
        subset=["sofascore_id", "season"],
        keep=False
    )
].sort_values(["sofascore_id", "season"])

,sofascore_id,season,market_value_08,market_value_09,market_value_10,market_value_11,market_value_12,market_value_01,market_value_02,market_value_03,market_value_04,market_value_05,market_value_06,market_value_07
17917,7635.0,2023-2024,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,3500000,3500000
17918,7635.0,2023-2024,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,4000000,3500000,3500000
973,7635.0,2024-2025,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000
974,7635.0,2024-2025,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000
20351,7635.0,2025-2026,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000,3500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5683,2059250.0,2023-2024,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4844,2059250.0,2024-2025,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4845,2059250.0,2024-2025,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6360,2059250.0,2025-2026,<NA>,<NA>,<NA>,<NA>,100000,100000,100000,100000,100000,700000,700000,700000


In [194]:
market_values_mapping = (
    market_values_mapping
    .drop_duplicates(subset=["sofascore_id", "season"])
    .reset_index(drop=True)
)

In [195]:
market_values_mapping.duplicated(
    subset=["sofascore_id", "season"]
).sum()

np.int64(0)

In [228]:
duplicated_mv = market_values_mapping[
    market_values_mapping.duplicated(
        subset=["sofascore_id", "season"],
        keep=False
    )
]

duplicated_mv.groupby(
    ["sofascore_id", "season"]
).nunique().max(axis=1).value_counts()

Series([], Name: count, dtype: int64)

Limpieza de mapping transfers

In [201]:
transfers = con.sql("""
    SELECT *
    FROM player_season_transfers
""").df()

In [202]:
transfers_mapping = (
    matched_mapping[
        ["sofascore_id", "player_id"]
    ]
    .drop_duplicates()
    .merge(
        transfers,
        on="player_id",
        how="left"
    )
)

In [203]:
transfers_mapping = transfers_mapping[
    [
        "sofascore_id",
        "season",
        "transferred",
        "transfer_count",
        "transfer_fee_total",
        "transfer_fee_max"
    ]
]

In [204]:
transfers_mapping.duplicated(
    subset=["sofascore_id", "season"]
).sum()

np.int64(6)

In [205]:
transfers_mapping[
    transfers_mapping.duplicated(
        subset=["sofascore_id", "season"],
        keep=False
    )
].sort_values(["sofascore_id", "season"])

,sofascore_id,season,transferred,transfer_count,transfer_fee_total,transfer_fee_max
35114,942836.0,2021-2022,True,2.0,0.0,0.0
35177,942836.0,2021-2022,True,3.0,0.0,0.0
35117,942836.0,2023-2024,True,3.0,0.0,0.0
35178,942836.0,2023-2024,True,1.0,0.0,0.0
35118,942836.0,2024-2025,True,1.0,1500000.0,1500000.0
35180,942836.0,2024-2025,True,1.0,300000.0,300000.0
7040,1046795.0,2020-2021,True,2.0,0.0,0.0
36653,1046795.0,2020-2021,True,1.0,NaN,NaN
7042,1046795.0,2022-2023,True,2.0,0.0,0.0
36651,1046795.0,2022-2023,True,3.0,6500000.0,6500000.0


In [209]:
ambiguous_transfers = (
    transfers_mapping[
        transfers_mapping.duplicated(
            subset=["sofascore_id", "season"],
            keep=False
        )
    ]
    .sort_values(["sofascore_id", "season"])
)

In [213]:
ambiguous_ids = ambiguous_transfers["sofascore_id"].unique()

In [214]:
ambiguous_transfers[
    ["sofascore_id", "season",
     "transfer_count", "transfer_fee_total", "transfer_fee_max"]
].sort_values(["sofascore_id", "season"])

,sofascore_id,season,transfer_count,transfer_fee_total,transfer_fee_max
35114,942836.0,2021-2022,2.0,0.0,0.0
35177,942836.0,2021-2022,3.0,0.0,0.0
35117,942836.0,2023-2024,3.0,0.0,0.0
35178,942836.0,2023-2024,1.0,0.0,0.0
35118,942836.0,2024-2025,1.0,1500000.0,1500000.0
35180,942836.0,2024-2025,1.0,300000.0,300000.0
7040,1046795.0,2020-2021,2.0,0.0,0.0
36653,1046795.0,2020-2021,1.0,NaN,NaN
7042,1046795.0,2022-2023,2.0,0.0,0.0
36651,1046795.0,2022-2023,3.0,6500000.0,6500000.0


In [215]:
# Eliminamos combinaciones ambiguas

transfers_mapping = transfers_mapping[
    ~transfers_mapping.duplicated(
        subset=["sofascore_id", "season"],
        keep=False
    )
].copy()

In [217]:
# Eliminamos registros sin valor en temporada

transfers_mapping = transfers_mapping[
    transfers_mapping["season"].notna()
].copy()

## **Creación del dataset maestro**

In [232]:
player_info_master = player_info_mapping[
    [
        "sofascore_id",
        "date_of_birth",
        "birth_year",
        "country_of_citizenship",
        "country_of_birth"
    ]
].copy()

In [234]:
master_dataset = df_all_comps_processed.merge(
    player_info_master,
    on="sofascore_id",
    how="left"
)

In [235]:
master_dataset = master_dataset.merge(
    market_values_mapping,
    on=["sofascore_id", "season"],
    how="left"
)

In [236]:
master_dataset = master_dataset.merge(
    transfers_mapping,
    on=["sofascore_id", "season"],
    how="left"
)

##### Validación del master_dataset

In [240]:
master_dataset[master_dataset.duplicated()]

,player,season,games,minutes,goals,assists,shots,key_passes,yellow_cards,red_cards,...,market_value_02,market_value_03,market_value_04,market_value_05,market_value_06,market_value_07,transferred,transfer_count,transfer_fee_total,transfer_fee_max


In [243]:
len(master_dataset[master_dataset['minutes'].isna()])

0

In [244]:
nan_dict = {}

for col in master_dataset.columns.to_list():
    nan_dict[col] = len(master_dataset[master_dataset[col].isna()])
    
nan_dict

{'player': 0,
 'season': 0,
 'games': 0,
 'minutes': 0,
 'goals': 0,
 'assists': 0,
 'shots': 0,
 'key_passes': 0,
 'yellow_cards': 0,
 'red_cards': 0,
 'npg': 0,
 'xg': 0,
 'xag': 0,
 'npxg': 0,
 'xg_chain': 0,
 'xg_buildup': 0,
 'ninety_s': 0,
 'sofascore_rating_total': 0,
 'sofascore_rating_count': 0,
 'totw_appearances': 0,
 'starts': 0,
 'tackles': 0,
 'tackles_won': 0,
 'interceptions': 0,
 'clearances': 0,
 'blocked_shots': 0,
 'outfield_blocks': 0,
 'errors_leading_to_goal': 0,
 'errors_leading_to_shot': 0,
 'dribbled_past': 0,
 'aerials_won': 0,
 'aerials_lost': 0,
 'dribbles_completed': 0,
 'dribbles_attempted': 0,
 'ground_duels_won': 0,
 'duels_won': 0,
 'duels_lost': 0,
 'passes_total': 0,
 'passes_completed': 0,
 'passes_inaccurate': 0,
 'passes_final_third': 0,
 'passes_opp_half': 0,
 'passes_own_half': 0,
 'passes_opp_half_total': 0,
 'passes_own_half_total': 0,
 'long_balls_total': 0,
 'long_balls_completed': 0,
 'crosses_total': 0,
 'crosses_completed': 0,
 'chipped_p

In [245]:
master_dataset[
    master_dataset["transferred"].isna()
][
    [
        "player",
        "season",
        "sofascore_id",
        "transferred",
        "transfer_count",
        "transfer_fee_total",
        "transfer_fee_max"
    ]
].head(20)

,player,season,sofascore_id,transferred,transfer_count,transfer_fee_total,transfer_fee_max
1,Andre Dozzell,2023-2024,826133.0,NaN,NaN,NaN,NaN
3,Cody Drameh,2023-2024,991577.0,NaN,NaN,NaN,NaN
4,Dion Sanderson,2023-2024,970407.0,NaN,NaN,NaN,NaN
6,Emmanuel Longelo,2023-2024,991612.0,NaN,NaN,NaN,NaN
7,Ethan Laird,2023-2024,927363.0,NaN,NaN,NaN,NaN
8,Gary Gardner,2023-2024,177415.0,NaN,NaN,NaN,NaN
9,George Hall,2023-2024,1136413.0,NaN,NaN,NaN,NaN
10,Ivan Šunjić,2023-2024,283833.0,NaN,NaN,NaN,NaN
12,John Ruddy,2023-2024,1131.0,NaN,NaN,NaN,NaN
13,Jordan James,2023-2024,1136409.0,NaN,NaN,NaN,NaN


Para los nulos resultantes del LEFT JOIN de la tabla transfers, podemos determinar que el valor es nulo por no haber encontrado transferencias de dichos jugadores durante las temporadas analizadas, por lo que determinamos False como el valor de `transferred` y 0 para el resto de variables relacionadas con los importes o cantidad de transferencias.

In [246]:
master_dataset["transferred"] = (
    master_dataset["transferred"]
    .fillna(False)
    .astype(bool)
)

master_dataset["transfer_count"] = (
    master_dataset["transfer_count"]
    .fillna(0)
)

master_dataset["transfer_fee_total"] = (
    master_dataset["transfer_fee_total"]
    .fillna(0)
)

master_dataset["transfer_fee_max"] = (
    master_dataset["transfer_fee_max"]
    .fillna(0)
)

In [249]:
master_dataset.to_parquet(
    "../data/processed_data/master_dataset.parquet",
    index=False
)